# SpaceX API — data collection
Collect historical Falcon 9 launches with the SpaceX REST API. Run in Google Colab. Outputs are saved to `data/`. The dataset and API evolve; record the extraction date.

In [ ]:
import requests, pandas as pd, numpy as np
from pathlib import Path
DATA=Path('../data'); DATA.mkdir(exist_ok=True)
url='https://api.spacexdata.com/v4/launches/past'
r=requests.get(url,timeout=45); r.raise_for_status(); launches=r.json()
print('Launch records from API:',len(launches))

In [ ]:
# API v4 returns document IDs; join rockets, launchpads and payloads to make a readable table.
s=requests.Session(); cache={}
def get_entity(collection,entity_id):
    if not entity_id: return {}
    key=(collection,entity_id)
    if key not in cache:
        resp=s.get(f'https://api.spacexdata.com/v4/{collection}/{entity_id}',timeout=30)
        resp.raise_for_status();cache[key]=resp.json()
    return cache[key]
rows=[]
for x in launches:
    rocket=get_entity('rockets',x.get('rocket'))
    if 'falcon 9' not in rocket.get('name','').lower(): continue
    pad=get_entity('launchpads',x.get('launchpad'))
    cores=x.get('cores') or []
    core=cores[0] if cores else {}
    payloads=[get_entity('payloads',p) for p in x.get('payloads',[])]
    rows.append({'FlightNumber':x.get('flight_number'),'Date':x.get('date_utc'),
        'Rocket':rocket.get('name'),'LaunchSite':pad.get('name'),
        'Latitude':pad.get('latitude'),'Longitude':pad.get('longitude'),
        'PayloadMass':sum(p.get('mass_kg') or 0 for p in payloads),
        'Orbit':payloads[0].get('orbit') if payloads else None,
        'Flights':core.get('flight'),'GridFins':core.get('gridfins'),
        'Reused':core.get('reused'),'Legs':core.get('legs'),
        'LandingOutcome':core.get('landing_success'),
        'LandingType':core.get('landing_type'),
        'LaunchSuccess':x.get('success'),'LaunchID':x.get('id')})
df=pd.DataFrame(rows).sort_values('Date').reset_index(drop=True)
df.to_csv(DATA/'spacex_api.csv',index=False)
print(df.shape);display(df.head())

**Method note:** The course historical dataset and current API may have different date ranges and field definitions. Use the course-compatible dataset for reproducing the IBM rubric; treat the API extraction as a separate source.